In [ ]:
import time
import multiprocessing as mp
from dataclasses import dataclass
from itertools import product

import numpy as np
import torch
import pandas as pd

from puzzle_solvers.solvers.gradient_descent.state import GDSGDState, GDAdamState
from puzzle_solvers.solvers.gradient_descent.strategy import GDStrategy
from puzzle_solvers.losses import ClueReconstructionLoss, LineAutomatonLoss
from puzzle_solvers.aggregation import FirstAggregator
from puzzle_solvers.solver import Solver
from puzzle_solvers.types import StoppingReason
from puzzle_solvers.generate_traces.factory import build_solvers
from puzzle_solvers.generate_traces.runner import _iter_puzzle_args
from neural_models.dataset import NonogramsDataset
from puzzle_solvers.generate_traces.config import resolve
from puzzle_solvers.solvers.utils.const import DEVICE
from utils.notebook_utils import reset_cwd_and_env

reset_cwd_and_env()


In [ ]:

print(f"Device: {DEVICE}")

# Load test puzzles (first 10 from 5x5 test set)
dataset = NonogramsDataset(
    file_name="5x5-100_000-val.jsonl",
    max_size=100,
    padding_type="positional"
)
print(f"Loaded {len(dataset)} puzzles")

@dataclass
class SweepConfig:
    name: str
    optimizer: str  # "sgd" or "adam"
    loss: str       # "clue_reconstruction" or "line_automata"
    lr: float
    init_scale: float = 0.1
    momentum: float = 0.0
    nesterov: bool = False
    betas: tuple = (0.9, 0.999)
    eps: float = 1e-8
    temp_start: float = 1.0
    temp_end: float = 0.5
    aux_weight: float = 0.0
    binarize_weight: float = 0.0
    line_reduction: str = "sum"
    max_steps: int = 1000
    stopping_threshold: float = 1e-5
    solved_threshold: float = 0.2
    k: int = 1

def make_solver(cfg: SweepConfig) -> Solver:
    if cfg.loss == "clue_reconstruction":
        loss_fn = ClueReconstructionLoss(
            temperature_start=cfg.temp_start,
            temperature_end=cfg.temp_end,
            aux_weight=cfg.aux_weight,
            binarize_weight=cfg.binarize_weight,
        )
    else:
        loss_fn = LineAutomatonLoss(
            reduction="mean",
            line_reduction=cfg.line_reduction,
        )
    
    if cfg.optimizer == "sgd":
        state = GDSGDState(
            k=cfg.k, lr=cfg.lr, momentum=cfg.momentum,
            nesterov=cfg.nesterov,
        )
    else:
        state = GDAdamState(
            k=cfg.k, lr=cfg.lr, betas=cfg.betas,
            eps=cfg.eps,
        )
    
    strategy = GDStrategy(loss_fn=loss_fn)
    aggregator = FirstAggregator(unknown_handling="replace")
    
    return Solver(
        name=cfg.name,
        strategy=strategy,
        state=state,
        aggregator=aggregator,
        stopping_threshold=cfg.stopping_threshold,
        solved_threshold=cfg.solved_threshold,
    )


In [ ]:

# Define sweep configurations (~28 configs)
configs = []

# SGD + ClueReconstructionLoss
for lr in []:
    for temp in [0.75, 1.5]:
        configs.append(SweepConfig(
            name=f"sgd_clue_lr{lr}_temp{temp}",
            optimizer="sgd", loss="clue_reconstruction",
            lr=lr, temp_start=temp, temp_end=0.5,
            aux_weight=0.0, binarize_weight=0.0, max_steps=2000,
        ))

# SGD + LineAutomatonLoss
for lr in [0.8, 0.9, 1.0, 1.1, 1.2]:
    configs.append(SweepConfig(
        name=f"sgd_auto_lr{lr}",
        optimizer="sgd", loss="line_automata",
        lr=lr, line_reduction="sum", max_steps=2000,
    ))

# Adam + ClueReconstructionLoss
for lr in []:
    for temp in [2.0, 3.0, 4.0]:
        configs.append(SweepConfig(
            name=f"adam_clue_lr{lr}_temp{temp}",
            optimizer="adam", loss="clue_reconstruction",
            lr=lr, temp_start=temp, temp_end=0.5,
            aux_weight=0.0, binarize_weight=0.0, max_steps=2000,
        ))

# Adam + LineAutomatonLoss
for lr in []:
    configs.append(SweepConfig(
        name=f"adam_auto_lr{lr}",
        optimizer="adam", loss="line_automata",
        lr=lr, line_reduction="sum", max_steps=2000,
    ))

print(f"Total configs: {len(configs)}")

# Build all solvers
solvers = {cfg.name: make_solver(cfg) for cfg in configs}


In [ ]:

# Prepare puzzle arguments
puzzle_args = list(_iter_puzzle_args(dataset))
max_time = 1.0
seeds = [42]

def run_config(config_name: str, seed: int):
    solver = solvers[config_name]
    seeded = solver.with_seed(seed)
    
    solved = 0
    total_steps = 0
    total_time = 0.0
    reasons = {r.name: 0 for r in StoppingReason}
    
    for puzzle_idx, puzzle_dict, clues, grid in puzzle_args:
        start = time.perf_counter()
        steps, num_steps, reason = seeded.solve(clues=clues, solution=grid, max_time=max_time)
        elapsed = time.perf_counter() - start
        
        reasons[reason.name] += 1
        total_steps += num_steps
        total_time += elapsed
        if reason == StoppingReason.SOLVED:
            solved += 1
    
    return {
        "config": config_name,
        "seed": seed,
        "solved": solved,
        "total": len(puzzle_args),
        "solve_rate": solved / len(puzzle_args),
        "avg_steps": total_steps / len(puzzle_args),
        "avg_time": total_time / len(puzzle_args),
        "reasons": reasons,
    }

# Run sweep
config_names = list(solvers.keys())
tasks = list(product(config_names, seeds))
print(f"Total tasks: {len(tasks)}")

results = []
for i, (config_name, seed) in enumerate(tasks):
    print(f"[{i+1}/{len(tasks)}] {config_name} seed={seed}", end=" ", flush=True)
    r = run_config(config_name, seed)
    results.append(r)
    print(f"→ solve_rate={r['solve_rate']:.2f}, avg_steps={r['avg_steps']:.1f}, avg_time={r['avg_time']:.2f}s")


In [ ]:

# Aggregate results
df = pd.DataFrame(results)
agg = df.groupby("config").agg({
    "solve_rate": ["mean", "std"],
    "avg_steps": "mean",
    "avg_time": "mean",
}).round(4)
agg.columns = ["_".join(col).strip() for col in agg.columns.values]
agg = agg.reset_index().sort_values("solve_rate_mean", ascending=False)

print("\n=== RESULTS (averaged over seeds) ===")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(agg.to_string(index=False))

# By optimizer/loss
df["optimizer"] = df["config"].str.split("_").str[0]
df["loss"] = df["config"].str.split("_").str[1]
summary = df.groupby(["optimizer", "loss"])["solve_rate"].agg(["mean", "std", "max"]).round(4)
print("\n=== BY OPTIMIZER × LOSS ===")
print(summary.to_string())

# Top 5
print("\n=== TOP 5 CONFIGS ===")
top5 = agg.head(5)[["config", "solve_rate_mean", "solve_rate_std", "avg_steps_mean", "avg_time_mean"]]
print(top5.to_string(index=False))

# Final eval on best config
if len(agg) > 0:
    best_config = agg.iloc[0]["config"]
    print(f"\nBest config: {best_config}")
    
    dataset_full = NonogramsDataset(
        file_name="5x5-200_000-test.jsonl",
        max_size=50,
        padding_type="positional"
    )
    puzzle_args_full = list(_iter_puzzle_args(dataset_full))
    solver = solvers[best_config].with_seed(42)
    
    solved = 0
    total_steps = 0
    reasons = {r.name: 0 for r in StoppingReason}
    
    for puzzle_idx, puzzle_dict, clues, grid in puzzle_args_full:
        steps, num_steps, reason = solver.solve(clues=clues, solution=grid, max_time=5.0)
        reasons[reason.name] += 1
        total_steps += num_steps
        if reason == StoppingReason.SOLVED:
            solved += 1
    
    print(f"\nFinal eval on {len(puzzle_args_full)} puzzles:")
    print(f"  Solved: {solved}/{len(puzzle_args_full)} ({100*solved/len(puzzle_args_full):.1f}%)")
    print(f"  Avg steps: {total_steps/len(puzzle_args_full):.1f}")
    print(f"  Reasons: {reasons}")